# Hugging Face Upload

This notebook is intentionally separate from the numbered `01`-`05` pipeline. Use it only after training/evaluation is finished and you want to package the released model for Hugging Face Hub.

Default behavior:
- selects the best fold from `results/kfold_metrics.csv`
- exports a clean release bundle from the chosen checkpoint
- writes a model card linked back to the GitHub repo
- optionally uploads the bundle to Hugging Face Hub


In [3]:
import sys, os, json
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from huggingface_hub import HfApi
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src import config as C


In [4]:
REPO_ID = 'vulonviing/roberta-babe-baseline'
GITHUB_URL = 'https://github.com/vulonviing/babe-roberta-baseline'
EXPORT_DIR = C.MODELS_DIR / '_hf_export' / 'best_fold'
UPLOAD = True
PRIVATE = False

# Run this once in a terminal before setting UPLOAD = True:
# hf auth login

print(f'Repo: {REPO_ID}')
print(f'Export dir: {EXPORT_DIR}')
print(f'Upload enabled: {UPLOAD}')


Repo: vulonviing/roberta-babe-baseline
Export dir: /Users/emrecanulu/Documents/Media Bias Application/Project 1: BABE Baseline/models/_hf_export/best_fold
Upload enabled: True


In [5]:
kf = pd.read_csv(C.RESULTS_DIR / 'kfold_metrics.csv')
best_row = kf.sort_values('eval_f1_macro', ascending=False).iloc[0]

BEST_FOLD = int(best_row['fold'])
BEST_SCORE = float(best_row['eval_f1_macro'])
fold_dir = C.MODELS_DIR / f'fold_{BEST_FOLD}'
checkpoints = sorted(
    [p for p in fold_dir.iterdir() if p.is_dir() and p.name.startswith('checkpoint-')],
    key=lambda p: int(p.name.split('-')[-1]),
)
if not checkpoints:
    raise FileNotFoundError(f'No checkpoint directories found under {fold_dir}')

CHECKPOINT_DIR = checkpoints[-1]
display(best_row.to_frame().T)
print(f'Selected checkpoint: {CHECKPOINT_DIR}')


,eval_loss,eval_accuracy,eval_f1_macro,eval_f1_biased,eval_precision_macro,eval_recall_macro,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,fold
0,0.493759,0.877589,0.876494,0.888124,0.875396,0.878364,3.751,141.564,4.532,4.0,0.0


Selected checkpoint: /Users/emrecanulu/Documents/Media Bias Application/Project 1: BABE Baseline/models/fold_0/checkpoint-532


In [6]:
with open(C.RESULTS_DIR / 'kfold_summary.json') as f:
    kfold_summary = json.load(f)

with open(C.RESULTS_DIR / 'final_test_metrics.json') as f:
    final_metrics = json.load(f)

def fmt(value):
    return f'{value:.3f}'

model_card = f"""---
library_name: transformers
pipeline_tag: text-classification
base_model: {C.MODEL_NAME}
tags:
- text-classification
- media-bias
- roberta
datasets:
- {C.HF_DATASET}
language:
- en
---

# {REPO_ID.split('/')[-1]}

Best-fold checkpoint from a 5-fold RoBERTa-base reproduction of BABE sentence-level media bias classification.

- Training code: [{GITHUB_URL}]({GITHUB_URL})
- Source dataset: [https://huggingface.co/datasets/{C.HF_DATASET}](https://huggingface.co/datasets/{C.HF_DATASET})
- Released checkpoint: `{CHECKPOINT_DIR.relative_to(C.ROOT)}`
- Selected checkpoint: `fold_{BEST_FOLD}` with macro-F1 `{BEST_SCORE:.3f}`
- Summary statement: trained on 80% of BABE, 5-fold CV mean: `{fmt(kfold_summary['mean']['eval_f1_macro'])} +- {fmt(kfold_summary['std']['eval_f1_macro'])}`

## Model details

| Item | Value |
|---|---|
| Base model | `{C.MODEL_NAME}` |
| Task | Sentence-level media bias classification |
| Labels | `non-biased`, `biased` |
| Max sequence length | `{C.MAX_LENGTH}` |
| Epochs | `{C.NUM_EPOCHS}` |
| Learning rate | `{C.LEARNING_RATE}` |
| Batch size | `{C.BATCH_SIZE}` train / `{C.EVAL_BATCH_SIZE}` eval |
| Weight decay | `{C.WEIGHT_DECAY}` |
| Warmup ratio | `{C.WARMUP_RATIO}` |
| Random seed | `{C.SEED}` |

## Cross-validation summary

| Metric | Mean +- Std |
|---|---|
| Macro-F1 | {fmt(kfold_summary['mean']['eval_f1_macro'])} +- {fmt(kfold_summary['std']['eval_f1_macro'])} |
| Accuracy | {fmt(kfold_summary['mean']['eval_accuracy'])} +- {fmt(kfold_summary['std']['eval_accuracy'])} |
| Precision (macro) | {fmt(kfold_summary['mean']['eval_precision_macro'])} +- {fmt(kfold_summary['std']['eval_precision_macro'])} |
| Recall (macro) | {fmt(kfold_summary['mean']['eval_recall_macro'])} +- {fmt(kfold_summary['std']['eval_recall_macro'])} |
| Biased F1 | {fmt(kfold_summary['mean']['eval_f1_biased'])} +- {fmt(kfold_summary['std']['eval_f1_biased'])} |

Per-fold macro-F1 values in the repo: `0.876, 0.854, 0.845, 0.852, 0.856`.

## Held-out quick-run reference

| Metric | Score |
|---|---|
| Macro-F1 | {fmt(final_metrics['f1_macro'])} |
| Accuracy | {fmt(final_metrics['accuracy'])} |
| Precision (macro) | {fmt(final_metrics['precision_macro'])} |
| Recall (macro) | {fmt(final_metrics['recall_macro'])} |
| Biased F1 | {fmt(final_metrics['f1_biased'])} |

Confusion matrix from the held-out quick run (`n=468`):

|  | Pred non-biased | Pred biased |
|---|---|---|
| True non-biased (207) | 180 | 27 |
| True biased (261) | 33 | 228 |

## Usage

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

repo_id = '{REPO_ID}'
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)
```
"""

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
model.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)
(EXPORT_DIR / 'README.md').write_text(model_card, encoding='utf-8')

print('Prepared files:')
for path in sorted(EXPORT_DIR.iterdir()):
    print('-', path.name)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

Prepared files:
- README.md
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json


In [7]:
if UPLOAD:
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=PRIVATE)
    api.upload_folder(
        folder_path=str(EXPORT_DIR),
        repo_id=REPO_ID,
        repo_type='model',
        commit_message='Upload best BABE fold checkpoint',
    )
    print(f'Uploaded to https://huggingface.co/{REPO_ID}')
else:
    print("Dry run only. Run `hf auth login` in a terminal, then set `UPLOAD = True` and re-run this cell.")


Processing Files (1 / 1): 100%|██████████|  499MB /  499MB, 13.7MB/s  
New Data Upload: 100%|██████████|  499MB /  499MB, 13.7MB/s  


Uploaded to https://huggingface.co/vulonviing/roberta-babe-baseline
